In [1]:
import torch
import torch.nn as nn
import numpy as np

In [2]:
data = np.load('./flight_data_for_rnn.npz')
X_train = data['X_train']
y_train = data['y_train']
anchor_train = data['anchor_train']
X_test = data['X_test']
y_test = data['y_test']
anchor_test = data['anchor_test']

In [3]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

'cuda'

In [4]:
X_train = torch.tensor(X_train, dtype=torch.float32).to(device)
y_train = torch.tensor(y_train, dtype=torch.float32).to(device)
anchor_train = torch.tensor(anchor_train, dtype=torch.float32).to(device)
X_test = torch.tensor(X_test, dtype=torch.float32).to(device)
y_test = torch.tensor(y_test, dtype=torch.float32).to(device)
anchor_test = torch.tensor(anchor_test, dtype=torch.float32).to(device)

In [5]:
from torch.utils.data import TensorDataset, DataLoader
train_ds = TensorDataset(X_train, y_train, anchor_train)
test_ds = TensorDataset(X_test, y_test, anchor_test)
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=64, shuffle=False)

In [13]:
class LSTMModelV1(nn.Module):
    def __init__(self, input_size=11, hidden_size=64, output_size=2, num_layers=2):
        super(LSTMModelV1, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        
        self.lstm = nn.LSTM(input_size=input_size,
                            hidden_size=hidden_size,
                            num_layers=num_layers,
                            batch_first=True,
                            dropout=0.3)
        
        self.fc1 = nn.Linear(hidden_size, 32)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(32, output_size)
        
        nn.init.uniform_(self.fc2.weight, -0.001, 0.001)
        nn.init.zeros_(self.fc2.bias)
        
    def forward(self, X):
        lstm_out, (h_n, c_n) = self.lstm(X)
        last_output = lstm_out[:, -1, :]
        
        x = self.fc1(last_output)
        x = self.relu(x)
        output = self.fc2(x)
        return output

model = LSTMModelV1().to(device)

In [ ]:
class HaversineLoss(nn.Module):
    def __init__(self):
        super(HaversineLoss, self).__init__()
        self.R = 6371000.0 # Earth's radius in meters
        
    def forward(self, pred_deltas, true_deltas, anchor_pos):
        pred_pos = anchor_pos + pred_deltas
        true_pos = anchor_pos + true_deltas
        
        lat1 = torch.deg2rad(pred_pos[:, 0])
        lon1 = torch.deg2rad(pred_pos[:, 1])
        lat2 = torch.deg2rad(true_pos[:, 0])
        lon2 = torch.deg2rad(true_pos[:, 1])
        
        dlat = lat2 - lat1
        dlon = lon2 - lon1
        
        a = torch.sin(dlat / 2)**2 + torch.cos(lat1) * torch.cos(lat2) * torch.sin(dlon / 2)**2
        
        eps = 1e-7
        a = torch.clamp(a, eps, 1.0 -eps)
        
        c = 2 * torch.asin(torch.sqrt(a))
        distances = self.R *c
        
        return distances.mean()